# Zynthe Dual-T4 Testing - Kaggle Notebook

**Hardware:** 2x NVIDIA T4 (16GB VRAM each)
**Testing:** GPT-2-medium → GPT-2 on CodeSearchNet with Feature Distillation
**Runtime:** ~5-10 minutes

## Setup

In [ ]:
# Check GPU setup
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  CUDA:{i} - {torch.cuda.get_device_name(i)}')
    print(f'    Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate
!pip install -q pyyaml omegaconf rich
!pip install -q matplotlib seaborn tqdm

## Clone Repository

In [ ]:
# REPLACE with your GitHub username
GITHUB_USERNAME = "YOUR_USERNAME"  # <-- CHANGE THIS
!git clone https://github.com/{GITHUB_USERNAME}/zynthe.git
%cd zynthe

## Quick Test (DistilGPT2 self-distillation)

In [ ]:
# Run quick test first - DistilGPT2 on wikitext
!python app/main.py distill --config configs/kaggle_quick_test.yaml

## Full Test (GPT-2-medium → GPT-2 on CodeSearchNet)

In [ ]:
# Run full test with dual T4: teacher on CUDA:0, student on CUDA:1
!python app/main.py distill --config configs/kaggle_dual_t4.yaml \
  --override train.epochs=1 data.train_samples=500

## Validate Results

In [ ]:
from pathlib import Path
import json

experiments_dir = Path('experiments')
if experiments_dir.exists():
    latest = sorted(experiments_dir.iterdir())[-1]
    print(f'Experiment: {latest.name}')
    
    # Check artifacts
    artifacts = ['config.yaml', 'student_model', 'training_metrics.json']
    for art in artifacts:
        exists = (latest / art).exists()
        print(f'  {art}: {"✅" if exists else "❌"}')
    
    # Show training metrics
    metrics_file = latest / 'training_metrics.json'
    if metrics_file.exists():
        with open(metrics_file) as f:
            metrics = json.load(f)
        print('\nMetrics:')
        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                print(f'  {k}: {v:.4f}')
else:
    print('No experiments found')

## GPU Memory Usage

In [ ]:
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved = torch.cuda.memory_reserved(i) / 1e9
    print(f'GPU {i}: Allocated={allocated:.2f}GB, Reserved={reserved:.2f}GB')